<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
%pip install -U -q keras-hub keras

In [ ]:
import keras
import keras_hub
import numpy as np

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_270m")

In [ ]:
gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

In [ ]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to pandas and sample
df = ds.to_pandas().sample(300, random_state=42)

features = {
    "prompts": df["input"].tolist(),
    "responses": df["output"].tolist()
}

In [ ]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

In [ ]:
gemma_lm.backbone.enable_lora(rank=4)

gemma_lm.preprocessor.sequence_length = 256

In [ ]:
gemma_lm.fit(features, epochs=1, batch_size=1)

In [ ]:
gemma_lm.generate("I have been having headaches every day for the past week. What could be causing this and what should I do?", max_length=500)

In [ ]:
gemma_lm.load_lora_weights("lora_weights.h5")